# ANFIS Breast Cancer Training (Paper-Style)

Notebook huấn luyện ANFIS bám sát paper:
- Dữ liệu WBCD (UCI original)
- Chọn đặc trưng theo PCA ranking trong paper: v1, v2, v3
- Chia dữ liệu đúng paper: 200 train / 263 checking / 200 test
- ANFIS dùng Gaussian MF (3 mỗi input => 27 rules)
- Huấn luyện backprop 300 epoch bằng `scikit-anfis`
- Lưu training loss log theo epoch

In [8]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from skanfis import scikit_anfis
from skanfis.membership import make_gauss_mfs
from skanfis.experimental import RMSELoss

print("Torch:", torch.__version__)

Torch: 2.12.0+cpu


In [9]:
# 1) Load WBCD goc (UCI Breast Cancer Wisconsin Original)
uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

cols = [
    "sample_code_number",
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
    "class",
]

df = pd.read_csv(uci_url, header=None, names=cols)

# Missing value trong bo du lieu nay nam o cot bare_nuclei, duoc ma hoa bang '?'
df = df.replace("?", np.nan).dropna().copy()
df["bare_nuclei"] = df["bare_nuclei"].astype(int)

# Class goc: benign=2, malignant=4 -> doi ve 0/1
df["target"] = (df["class"] == 4).astype(int)

feature_cols = [
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
]

X = df[feature_cols].values.astype(np.float32)
y = df["target"].values.astype(np.float32)

print("Dataset shape:", X.shape)
print("Class balance (malignant rate):", y.mean().round(4))

Dataset shape: (683, 9)
Class balance (malignant rate): 0.3499


In [10]:
# 2) Tien xu ly va chia tap BAM SAT paper
# Paper ANFIS: dung PCA de chon 3 dac trung goc dau tien (v1, v2, v3)
# va chia du lieu thanh 200 train / 263 checking / 200 test.

# Normalize toan bo 9 dac trung (paper co neu normalized data)
scaler = StandardScaler()
X_norm = scaler.fit_transform(X).astype(np.float32)

# Fit PCA tren 9 dac trung de phuc vu bao cao (feature selection theo paper)
pca_full = PCA(n_components=9, random_state=42)
pca_full.fit(X_norm)

selected_feature_names = [
    "clump_thickness",              # v1
    "uniformity_of_cell_size",      # v2
    "uniformity_of_cell_shape",     # v3
]
selected_idx = [feature_cols.index(c) for c in selected_feature_names]
X_selected = X_norm[:, selected_idx].astype(np.float32)

# Paper ghi 200/263/200 => tong 663 mau, nen lay dung 663 mau
rng = np.random.default_rng(42)
indices = rng.permutation(len(X_selected))[:663]
X_663 = X_selected[indices]
y_663 = y[indices]

X_train = X_663[:200]
y_train = y_663[:200]

X_check = X_663[200:463]
y_check = y_663[200:463]

X_test = X_663[463:663]
y_test = y_663[463:663]

print("Split theo paper (train/check/test):", X_train.shape, X_check.shape, X_test.shape)
print("So mau bi bo de bam paper:", int(len(X_selected) - len(X_663)))
print("PCA explained variance ratio (9 PCs):", np.round(pca_full.explained_variance_ratio_, 4))

Split theo paper (train/check/test): (200, 3) (263, 3) (200, 3)
So mau bi bo de bam paper: 20
PCA explained variance ratio (9 PCs): [0.6555 0.0862 0.0599 0.0511 0.0423 0.0335 0.0327 0.029  0.0098]


In [11]:
# 3) Train ANFIS theo paper + ghi training loss log tung epoch
epochs = 300
learning_rate = 0.01  # user chot

# Grid partitioning: 3 Gaussian MF moi input -> 3^3 = 27 rules
invars = []
for i, feat_name in enumerate(selected_feature_names):
    col = X_train[:, i]
    col_min, col_max = float(col.min()), float(col.max())
    centers = np.linspace(col_min, col_max, 3).tolist()
    sigma = max((col_max - col_min) / 3.0, 1e-3)
    invars.append((feat_name, make_gauss_mfs(sigma=sigma, mu_list=centers)))

model = scikit_anfis(
    data=invars,
    description="WBCD_PaperStyle_ANFIS_PCAFeatureSelection",
    epoch=epochs,
    hybrid=True,  # backprop-only theo paper mo ta
    label="c",
)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
criterion = RMSELoss()

X_train_t = torch.from_numpy(X_train).float()
y_train_t = torch.from_numpy(y_train).float().unsqueeze(-1)

history = []
best_rmse = float("inf")
best_ckpt = Path("tmp.pkl")

model.train()
model.is_training = True

for ep in range(1, epochs + 1):
    y_pred = model(X_train_t, y_train_t)

    mse = torch.nn.functional.mse_loss(y_pred, y_train_t).item()
    rmse = float(np.sqrt(mse))

    optimizer.zero_grad()
    loss = criterion(y_pred, y_train_t)
    loss.backward()
    optimizer.step()

    history.append(
        {
            "epoch": ep,
            "mse": mse,
            "rmse": rmse,
            "loss": float(loss.item()),
            "lr": optimizer.param_groups[0]["lr"],
        }
    )

    if rmse < best_rmse:
        best_rmse = rmse
        model.save(str(best_ckpt))

    if ep % 25 == 0 or ep == 1:
        print(f"Epoch {ep:03d}/{epochs} - RMSE: {rmse:.6f}")

# Load lai best checkpoint de predict
model.load(str(best_ckpt))
model.eval()
model.is_training = False

loss_log_df = pd.DataFrame(history)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path("models")
out_dir.mkdir(exist_ok=True)
loss_log_path = out_dir / f"{stamp}_paperstyle_pca_feature_select_training_loss_log.csv"
loss_log_df.to_csv(loss_log_path, index=False)

print("Best train RMSE:", round(best_rmse, 6))
print("Num rules:", model.num_rules)
print("Loss log saved:", loss_log_path)

Epoch 001/300 - RMSE: 0.553175
Epoch 025/300 - RMSE: 0.599999
Epoch 050/300 - RMSE: 0.599991
Epoch 075/300 - RMSE: 0.599956
Epoch 100/300 - RMSE: 0.599100
Epoch 125/300 - RMSE: 0.594772
Epoch 150/300 - RMSE: 0.599747
Epoch 175/300 - RMSE: 0.594269
Epoch 200/300 - RMSE: 0.295524
Epoch 225/300 - RMSE: 0.264718
Epoch 250/300 - RMSE: 0.257145
Epoch 275/300 - RMSE: 0.273413
Epoch 300/300 - RMSE: 0.274776
Best train RMSE: 0.256281
Num rules: 27
Loss log saved: models\20260618_192735_paperstyle_pca_feature_select_training_loss_log.csv


In [12]:
# 4) Evaluate tren check/test + tao bang metric de xuat JSON/CSV

def to_binary(pred):
    # scikit-anfis label='c' tra ve gia tri da lam tron, nhung van clip de an toan
    pred = np.asarray(pred).reshape(-1)
    pred = np.clip(np.round(pred), 0, 1).astype(int)
    return pred

# Checking set
y_check_pred_raw = model.predict(X_check)
y_check_pred = to_binary(y_check_pred_raw)

# Test set
y_test_pred_raw = model.predict(X_test)
y_test_pred = to_binary(y_test_pred_raw)


def evaluate_split(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_pred)
    except ValueError:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{name} metrics")
    print("- Accuracy :", round(acc, 4))
    print("- Precision:", round(prec, 4))
    print("- Recall   :", round(rec, 4))
    print("- F1-score :", round(f1, 4))
    print("- ROC-AUC  :", round(auc, 4) if not np.isnan(auc) else "nan")
    print("- Confusion matrix:\n", cm)

    return {
        "split": name,
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "roc_auc": None if np.isnan(auc) else float(auc),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "support": int(len(y_true)),
    }

check_metrics = evaluate_split("CHECK", y_check.astype(int), y_check_pred)
test_metrics = evaluate_split("TEST", y_test.astype(int), y_test_pred)

metrics_df = pd.DataFrame([check_metrics, test_metrics])
metrics_summary = {
    "CHECK": check_metrics,
    "TEST": test_metrics,
}


CHECK metrics
- Accuracy : 0.9582
- Precision: 0.9867
- Recall   : 0.881
- F1-score : 0.9308
- ROC-AUC  : 0.9377
- Confusion matrix:
 [[178   1]
 [ 10  74]]

TEST metrics
- Accuracy : 0.93
- Precision: 0.9286
- Recall   : 0.8784
- F1-score : 0.9028
- ROC-AUC  : 0.9193
- Confusion matrix:
 [[121   5]
 [  9  65]]


In [13]:
# 5) Luu artifact phuc vu bao cao
import json
import pickle

best_model_path = out_dir / f"{stamp}_paperstyle_pca_feature_select_best_model.pkl"
scaler_path = out_dir / f"{stamp}_paperstyle_scaler.pkl"
pca_info_path = out_dir / f"{stamp}_paperstyle_pca_info.pkl"
metrics_json_path = out_dir / f"{stamp}_paperstyle_metrics.json"
metrics_csv_path = out_dir / f"{stamp}_paperstyle_metrics.csv"

# Luu model ANFIS
model.save(str(best_model_path))

# Luu scaler & PCA object (de tai lap phan feature selection)
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
with open(pca_info_path, "wb") as f:
    pickle.dump(pca_full, f)

# Luu metric sang JSON va CSV de thong ke/bieu do
metrics_payload = {
    "timestamp": stamp,
    "run_tag": "paperstyle_pca_feature_select",
    "metrics": metrics_summary,
}
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

metrics_df.to_csv(metrics_csv_path, index=False)

meta = {
    "timestamp": stamp,
    "dataset": "UCI WBCD original",
    "n_features_raw": int(X.shape[1]),
    "feature_selection_method": "PCA-based ranking (paper), keep top-3 original features",
    "selected_features": selected_feature_names,
    "pca_explained_variance_ratio": pca_full.explained_variance_ratio_.tolist(),
    "split_mode": "paper strict 200/263/200",
    "used_samples": int(len(X_663)),
    "dropped_samples_for_paper_split": int(len(X_selected) - len(X_663)),
    "train_size": int(len(X_train)),
    "check_size": int(len(X_check)),
    "test_size": int(len(X_test)),
    "epoch": epochs,
    "optimizer": "SGD",
    "learning_rate": learning_rate,
    "membership_function": "Gaussian",
    "membership_per_input": 3,
    "num_rules": int(model.num_rules),
    "best_train_rmse": float(best_rmse),
    "loss_log_path": str(loss_log_path),
    "best_model_path": str(best_model_path),
    "metrics_json_path": str(metrics_json_path),
    "metrics_csv_path": str(metrics_csv_path),
}

meta_path = out_dir / f"{stamp}_paperstyle_meta.json"
pd.Series(meta).to_json(meta_path, indent=2)

print("Saved:")
print("-", best_model_path)
print("-", scaler_path)
print("-", pca_info_path)
print("-", metrics_json_path)
print("-", metrics_csv_path)
print("-", meta_path)

Saved:
- models\20260618_192735_paperstyle_pca_feature_select_best_model.pkl
- models\20260618_192735_paperstyle_scaler.pkl
- models\20260618_192735_paperstyle_pca_info.pkl
- models\20260618_192735_paperstyle_metrics.json
- models\20260618_192735_paperstyle_metrics.csv
- models\20260618_192735_paperstyle_meta.json


# Convert HTML

In [14]:
from datetime import datetime

html_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
html_trial_tag = "paperstyle_pca_feature_select"
html_split_tag = "ANFIS_-_split_200_263_200"
html_output_name = f"{html_timestamp}_{html_trial_tag}_{html_split_tag}.html"

log_dir = Path("nhat-ky")
log_dir.mkdir(parents=True, exist_ok=True)

!jupyter nbconvert --to html anfis_pca_scikit_anfis_training.ipynb --output {html_output_name} --output-dir ./nhat-ky

[NbConvertApp] Converting notebook anfis_pca_scikit_anfis_training.ipynb to html
[NbConvertApp] Writing 335144 bytes to nhat-ky\20260618_192735_paperstyle_pca_feature_select_ANFIS_-_split_200_263_200.html
